# 02 — Agent Evaluation

**PPE Compliance Agent — ITAI 1378 Final Project**

Evaluates the agent at two levels, per course requirements:

1. **Component level** — the CV model's own metrics (mAP, precision, recall) from training
2. **System level** — task success rate across 10-20+ test scenarios, robustness to bad inputs, efficiency, and honest failure analysis

**Run the Setup cell first, every time.** If anything gets confused, use **Runtime → Restart session**, then run Setup again from a clean start.

## 0. Setup — run this first, every session

In [ ]:
import os

REPO_URL = "https://github.com/Huynguyen-175/ITAI1378_Final_PPEComplianceAgent.git"
REPO_DIR = "/content/ITAI1378_Final_PPEComplianceAgent"

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Repo root:", os.getcwd())

!pip install -q -r requirements.txt

os.chdir(f"{REPO_DIR}/notebooks")
print("Now in:", os.getcwd())

try:
    import ultralytics
    print(f"\nultralytics OK — version {ultralytics.__version__}")
except ImportError as e:
    print(f"\nSETUP FAILED: {e}")
    print("Try running this Setup cell again, or run: !pip install -q ultralytics")

## 0b. Load real trained weights from Google Drive

Searches your whole Drive for `best_yolov8n_ppe.pt` — skip this cell if you haven't trained yet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, shutil

matches = glob.glob("/content/drive/MyDrive/**/best_yolov8n_ppe.pt", recursive=True)

if not matches:
    print("No file named 'best_yolov8n_ppe.pt' found anywhere in your Drive.")
    print("Skip this cell if you haven't trained yet — the agent will use fallback weights.")
else:
    src = matches[0]
    if len(matches) > 1:
        print(f"Found {len(matches)} matches, using the first one:")
        for m in matches:
            print(" ", m)
    os.makedirs("../models/trained", exist_ok=True)
    shutil.copy(src, "../models/trained/best_yolov8n_ppe.pt")
    print(f"\nCopied from: {src}")

## 1. Import the agent

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from agents.ppe_compliance_agent import PPEComplianceAgent

agent = PPEComplianceAgent(
    weights_path="../models/trained/best_yolov8n_ppe.pt",
    results_dir="../results",
)

## 2. Component-Level Metrics (from training)

In [ ]:
component_metrics = {
    "mAP@0.5": 0.900,
    "mAP@0.5:0.95": 0.608,
    "precision": 0.934,
    "recall": 0.825,
    "epochs_trained": 75,
    "training_time_min": 76.4,
}
for k, v in component_metrics.items():
    print(f"{k:20s}: {v}")

## 3. System-Level Evaluation

Two parts:

- **3a. Smoke test** — the 3 bundled sample images in `data/sample/`. Fast, useful for
  regression-testing changes (like the reasoning fix below), but NOT a statistically
  meaningful accuracy number on its own.
- **3b. Full evaluation** — 10-20+ images from the actual Roboflow test set, with ground
  truth built automatically from the YOLO label files. This is the number that actually
  satisfies the assignment's evaluation requirement.

### 3a. Smoke Test (3 bundled sample images)

Also doubles as a before/after regression check for the reasoning-layer confidence fix
(see `agents/reasoning.py` v2 and Section 6 below) — `caregiver_clinic_masked.jpg` used
to be misclassified before that fix.

In [ ]:
import os

SMOKE_TEST_GROUND_TRUTH = {
    "caregiver_car_nomask.jpg": "NON_COMPLIANT",
    "caregiver_home_nomask.jpg": "NON_COMPLIANT",
    "caregiver_clinic_masked.jpg": "COMPLIANT",
}

smoke_traces = agent.run("../data/sample")

correct, total = 0, 0
print(f"{'Image':35s} {'Ground Truth':15s} {'Predicted':15s} {'Correct'}")
for t in smoke_traces:
    fname = os.path.basename(t["image_path"])
    truth = SMOKE_TEST_GROUND_TRUTH.get(fname)
    if truth is None:
        continue
    predicted = t["status"]
    is_correct = predicted == truth
    correct += int(is_correct)
    total += 1
    print(f"{fname:35s} {truth:15s} {predicted:15s} {is_correct}")

print(f"\nSmoke test success rate: {correct}/{total} = {correct/max(total,1)*100:.1f}%")
print("(3-image smoke test only — see 3b below for the real evaluation.)")

### 3b. Full Evaluation (Roboflow test set, 10-20+ images)

Downloads the actual test split and builds ground truth automatically from the YOLO
label files — no manual typing required.

In [ ]:
import roboflow

roboflow.login()
rf = roboflow.Roboflow()
project = rf.workspace("agh-ett2f").project("mask-detection-yolov8")
# confirm the current version number on the Roboflow project page before running
dataset = project.version(16).download("yolov8")

TEST_DIR = f"{dataset.location}/test/images"
LABELS_DIR = f"{dataset.location}/test/labels"
print("Test images:", TEST_DIR)
print("Test labels:", LABELS_DIR)

In [ ]:
# Confirm the class-id -> class-name mapping BEFORE building ground truth —
# Roboflow's class order isn't always 0/1/2 in the order you'd expect.
!cat {dataset.location}/data.yaml

In [ ]:
import os

# Update this dict if the printed data.yaml above shows a different class order.
# 'with_mask' -> COMPLIANT; 'without_mask' and 'incorrectly_worn_mask' -> NON_COMPLIANT.
CLASS_ID_TO_STATUS = {
    0: "NON_COMPLIANT",  # incorrectly_worn_mask (confirm against data.yaml)
    1: "COMPLIANT",       # with_mask (confirm against data.yaml)
    2: "NON_COMPLIANT",  # without_mask (confirm against data.yaml)
}

GROUND_TRUTH = {}
for label_file in os.listdir(LABELS_DIR):
    if not label_file.endswith(".txt"):
        continue
    with open(os.path.join(LABELS_DIR, label_file)) as f:
        class_ids = [int(line.split()[0]) for line in f if line.strip()]
    if not class_ids:
        continue
    statuses = [CLASS_ID_TO_STATUS[c] for c in class_ids]
    # if ANY violation class present in the image, ground truth is NON_COMPLIANT
    truth = "NON_COMPLIANT" if "NON_COMPLIANT" in statuses else "COMPLIANT"
    image_name = label_file.replace(".txt", ".jpg")
    GROUND_TRUTH[image_name] = truth

print(f"Built ground truth for {len(GROUND_TRUTH)} images")
print(f"COMPLIANT: {sum(1 for v in GROUND_TRUTH.values() if v == 'COMPLIANT')}")
print(f"NON_COMPLIANT: {sum(1 for v in GROUND_TRUTH.values() if v == 'NON_COMPLIANT')}")

In [ ]:
import random

# Full test set can be large — sample 20 images for a manageable, still-meaningful run.
# Increase/decrease N as you like; assignment minimum is 10.
N = 20
random.seed(42)
sample_names = random.sample(list(GROUND_TRUTH.keys()), min(N, len(GROUND_TRUTH)))

# Copy just this sample into a scratch folder so the agent only processes these N images
import shutil
EVAL_DIR = "../data/_eval_sample"
os.makedirs(EVAL_DIR, exist_ok=True)
for name in sample_names:
    shutil.copy(os.path.join(TEST_DIR, name), os.path.join(EVAL_DIR, name))

print(f"Prepared {len(sample_names)} images for evaluation in {EVAL_DIR}")

In [ ]:
full_eval_traces = agent.run(EVAL_DIR)

In [ ]:
correct, total = 0, 0
mismatches = []
print(f"{'Image':35s} {'Ground Truth':15s} {'Predicted':15s} {'Correct'}")
for t in full_eval_traces:
    fname = os.path.basename(t["image_path"])
    truth = GROUND_TRUTH.get(fname)
    if truth is None:
        continue
    predicted = t["status"]
    is_correct = predicted == truth
    correct += int(is_correct)
    total += 1
    print(f"{fname:35s} {truth:15s} {predicted:15s} {is_correct}")
    if not is_correct:
        mismatches.append((fname, truth, predicted, t))

print(f"\n=== TASK SUCCESS RATE: {correct}/{total} = {correct/max(total,1)*100:.1f}% ===")
print(f"\n{len(mismatches)} mismatch(es) found — these are your failure-case candidates for Section 6.")
for fname, truth, predicted, t in mismatches:
    print(f"\n  {fname}: expected {truth}, got {predicted}")
    print(f"    rule fired: {t['reasoning']['rule_fired']}")
    print(f"    detections: {t['perception']['detections']}")

## 4. Robustness — Bad Input Handling

In [ ]:
import cv2
import numpy as np

os.makedirs("../data/robustness_test", exist_ok=True)

with open("../data/robustness_test/corrupt.jpg", "w") as f:
    f.write("this is not an image")

# Using cv2 instead of PIL here — avoids a known Colab/Pillow JPEG-encoder bug
# (TypeError: function takes at most 16 arguments). cv2 is already a core
# dependency used throughout the pipeline, so this stays consistent.
tiny_img = np.full((10, 10, 3), 255, dtype=np.uint8)
cv2.imwrite("../data/robustness_test/tiny.jpg", tiny_img)

blank_img = np.full((640, 640, 3), 255, dtype=np.uint8)
cv2.imwrite("../data/robustness_test/blank.jpg", blank_img)

robustness_traces = agent.run("../data/robustness_test")

for t in robustness_traces:
    print(f"{os.path.basename(t['image_path']):20s} -> {t['status']:25s} ({t['preprocessing']['reason']})")

**Result:** corrupt and undersized files are caught at preprocessing and marked `SKIPPED_INVALID_INPUT` rather than crashing the batch. A valid-but-blank image is processed normally and correctly returns `NO_DETECTION` — the agent abstains instead of guessing.

## 5. Efficiency

In [ ]:
latencies = [t["latency_sec"] for t in full_eval_traces if "latency_sec" in t]
if latencies:
    print(f"Average latency: {sum(latencies)/len(latencies):.3f} sec/image")
    print(f"Min: {min(latencies):.3f}s | Max: {max(latencies):.3f}s")
    print(f"Based on {len(latencies)} images from the full evaluation run.")

## 6. Honest Failure Analysis

**Required: at least 2 documented failure cases with explanation.**

### Failure Case 1 (confirmed, real, fixed): false positive overrides correct detections

**Image:** `data/sample/caregiver_clinic_masked.jpg` — two people, both actually wearing masks.

**What happened:** the agent correctly detected both real masks at high confidence (`with_mask` 0.90 and 0.87), but also produced a spurious third detection (`without_mask`, confidence only 0.55) on a small orange object clipped to one person's scrub top — not a face at all, an ID badge holder. The original reasoning logic treated *any* `without_mask` detection as an automatic override regardless of confidence, so this single low-confidence false positive flipped the entire frame's verdict from COMPLIANT to NON_COMPLIANT.

**Raw detections:**
```json
{"class_name": "with_mask", "confidence": 0.9033, "bbox_xyxy": [552.5, 146.2, 661.5, 281.1]},
{"class_name": "with_mask", "confidence": 0.8696, "bbox_xyxy": [262.7, 114.5, 355.5, 221.3]},
{"class_name": "without_mask", "confidence": 0.5475, "bbox_xyxy": [146.4, 252.8, 186.4, 306.4]}
```

**Fix applied and verified** (`agents/reasoning.py`, v2): violation classes now require confidence ≥ 0.65 (`VIOLATION_CONFIDENCE_FLOOR`) before they're allowed to override a `with_mask` detection in the same frame. Low-confidence violation candidates are still logged (`suppressed_detections` in every trace) for audit transparency; they just no longer unilaterally flip the result.

**Before/after (Section 3a smoke test):**

| | Before fix | After fix |
|---|---|---|
| `caregiver_clinic_masked.jpg` | ❌ NON_COMPLIANT (wrong) | ✅ COMPLIANT (correct) |
| 3-image smoke test success rate | 2/3 (66.7%) | 3/3 (100%) |

Regression-tested against real violation cases (0.87, 0.90 confidence) to confirm genuine violations still correctly trigger NON_COMPLIANT after the fix.

### Failure Case 2 (confirmed, real, visually verified): watermark logo misdetected as an unmasked face

**Image:** `maksssksksss106_png.rf.161f2ff0006c678a4b6c779dcb5de56f.jpg` (from the 20-image full evaluation run, Section 3b) — ground truth COMPLIANT.

**What happened:** the agent predicted NON_COMPLIANT, driven by a single `without_mask` detection at 0.7668 confidence — well above the 0.65 override floor, so this is NOT the same reasoning-layer bug as Failure Case 1; the reasoning logic behaved correctly given its input. The actual photo shows a single, large, clear side-profile portrait with the mask worn correctly (covering nose and mouth). Visually inspecting the annotated output confirms the false detection's bounding box lines up with a small decorative watermark/logo graphic in the image's top-left corner — not the person's face at all.

**Raw detection:**
```json
{"class_name": "without_mask", "confidence": 0.7668, "bbox_xyxy": [13.5, 17.7, 44.5, 66.0]}
```
Box coordinates (top-left corner of the frame, ~31×48px) match exactly where the watermark sits in the source image, not where the person's face is.

**Root cause:** this is the same *category* of error as Failure Case 1 — a spurious detection on a small, non-face graphical element, not a genuine face-reading error. Different object type (a watermark logo here vs. an ID badge in Case 1), same underlying pattern: the detector occasionally mistakes small, colorful, face-adjacent-shaped graphics for an unmasked face.

**This is a model-level limitation, not a reasoning bug** — the reasoning layer's 0.65 confidence floor (added after Case 1) did NOT catch this one, since the false detection's confidence (0.77) is genuinely high. A confidence threshold alone can't distinguish a confident-but-wrong detection from a confident-and-right one. **Possible fixes for future work:** train on more examples with watermarks/logos/badges present so the model learns to ignore them, or add a face-plausibility check (e.g. minimum size relative to image, aspect ratio, or position heuristics) as a second filter in the reasoning stage before acting on any violation-class detection.

### Summary

Two distinct, real failure cases were found and documented, both visually confirmed against the actual annotated output images:

1. **Reasoning-layer bug** (Case 1) — found, fixed, and verified. A low-confidence false positive (0.55) on an ID badge could override correct high-confidence detections. Fixed with a confidence floor for violation overrides.
2. **Perception-layer limitation** (Case 2) — found and root-caused, not yet fixed. A high-confidence false positive (0.77) on a watermark logo was NOT caught by the Case 1 fix, since confidence alone can't distinguish a confident-but-wrong detection from a genuine one. Same error *category* as Case 1 (spurious detection on a small non-face object), different specific cause — flagged as future work.

**Overall system-level result:** 19/20 (95.0%) task success rate on a real, randomly sampled 20-image test set with ground truth from actual dataset labels — see Section 3b.